In [16]:
import sys, torch, pandas as pd, sklearn
print("python:", sys.version)
print("torch:", torch.__version__)
print("mps available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)

python: 3.11.14 (main, Oct 21 2025, 18:27:30) [Clang 20.1.8 ]
torch: 2.10.0
mps available: True
pandas: 3.0.1
sklearn: 1.8.0


# Data Manipulation

In [17]:
df = pd.read_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-alladults.csv')

print("Number of samples:", len(df))
df.head()

Number of samples: 445834


,race,gender,age_at_visit,temperature,heartrate,resprate,sbp,dbp,o2sat,pain,unable,chiefcomplaint,acuity
0,Black/African American Only,0,30.0,98.3,128.0,16.0,149.0,79.0,100.0,10.0,0,"Carbuncle, furuncle, boil, cellulitis...",3
1,Black/African American Only,0,64.0,98.0,106.0,18.0,142.0,111.0,100.0,4.0,0,Painful urination,3
2,Blank,1,73.0,101.4,93.0,NaN,112.0,49.0,97.0,0.0,0,Other symptoms referable to the respi...,4
3,Black/African American Only,0,69.0,98.4,107.0,18.0,101.0,63.0,100.0,NaN,1,Other symptoms referable to the respi...,5
4,Blank,0,51.0,97.9,74.0,18.0,114.0,84.0,97.0,7.0,0,"Abdominal pain, cramps, spasms, NOS",3


In [18]:
selected_cols = ["chiefcomplaint", "acuity"]
print("Columns:", list(df.columns))
df = df[selected_cols].copy()

# some basic cleaning
print("Rows before cleaning:", len(df))
df["chiefcomplaint"] = df["chiefcomplaint"].astype(str).fillna("").str.strip() 
df = df[df["chiefcomplaint"].str.len() > 0].copy()  # only keep examples with a chief complaint

print("Rows after cleaning 'chiefcomplaint':", len(df))

df = df[df["acuity"].isin([1,2,3,4,5])].copy()
df["acuity"] = df["acuity"].astype(int)
print("Rows after cleaning 'acuity':", len(df))

print("Rows after cleaning:", len(df))
print("\nAcuity distribution (counts):")
print(df["acuity"].value_counts().sort_index())
print("\nAcuity distribution (proportion):")
print((df["acuity"].value_counts(normalize=True).sort_index()).round(4))

# Text length stats (characters + approx words)
lens = df["chiefcomplaint"].str.len()
words = df["chiefcomplaint"].str.split().map(len)

print("\nChief complaint length (chars):")
print(lens.describe(percentiles=[.5,.9,.95,.99]).round(2))
print("\nChief complaint length (words):")
print(words.describe(percentiles=[.5,.9,.95,.99]).round(2))

# print("\nUnique subjects:", df["subject_id"].nunique())

Columns: ['race', 'gender', 'age_at_visit', 'temperature', 'heartrate', 'resprate', 'sbp', 'dbp', 'o2sat', 'pain', 'unable', 'chiefcomplaint', 'acuity']
Rows before cleaning: 445834
Rows after cleaning 'chiefcomplaint': 445817
Rows after cleaning 'acuity': 445817
Rows after cleaning: 445817

Acuity distribution (counts):
acuity
1     24042
2    143264
3    239670
4     36619
5      2222
Name: count, dtype: int64

Acuity distribution (proportion):
acuity
1    0.0539
2    0.3214
3    0.5376
4    0.0821
5    0.0050
Name: proportion, dtype: float64

Chief complaint length (chars):
count    445817.00
mean         15.28
std           8.76
min           1.00
50%          13.00
90%          28.00
95%          33.00
99%          40.00
max         136.00
Name: chiefcomplaint, dtype: float64

Chief complaint length (words):
count    445817.00
mean          2.51
std           1.30
min           1.00
50%           2.00
90%           4.00
95%           5.00
99%           6.00
max          16.00
Name

In [19]:
from sklearn.model_selection import train_test_split

# note: keys 1-5 map to 0-4, which is 5 classes
label2id = {1:0, 2:1, 3:2, 4:3, 5:4}
id2label = {v:k for k,v in label2id.items()}
df["label"] = df["acuity"].map(label2id)

# standard random split: 80/10/10 (train/val/test)
train_df, test_df = train_test_split(df, test_size=0.10, random_state=42)

# reset indices
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# 0.1111111 of the remaining 90% ≈ 10% of total data
tr_df, val_df = train_test_split(train_df, test_size=0.1111111, random_state=43)  

# reset indices again
tr_df  = tr_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

def summarize_split(name, d):
    print(f"\n{name}: rows={len(d):,}")
    print(d["acuity"].value_counts(normalize=True).sort_index().round(4))

summarize_split("TRAIN", tr_df)
summarize_split("VAL", val_df)
summarize_split("TEST", test_df)

print("\nSplitting complete. All instances treated independently.")


TRAIN: rows=356,653
acuity
1    0.0541
2    0.3215
3    0.5374
4    0.0820
5    0.0050
Name: proportion, dtype: float64

VAL: rows=44,582
acuity
1    0.0543
2    0.3185
3    0.5399
4    0.0819
5    0.0053
Name: proportion, dtype: float64

TEST: rows=44,582
acuity
1    0.0518
2    0.3228
3    0.5372
4    0.0834
5    0.0048
Name: proportion, dtype: float64

Splitting complete. All instances treated independently.


In [8]:
# test_df
test_df.to_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/PythonFolder/clinical_test_df.csv', index=False)

# Single-Modality Text Models

## TF-IDF with Logistic Regression Model Performance

We used validation set to tune values for $C = \frac{1}{\lambda}$ and n-gram range. 
The best validation result was:
- $C = 0.1$
- $\text{ngram\_range} = (1,3)$

which received a macro-F1 = $0.3794$

### Training and Tuning the Model

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score
from sklearn.metrics import confusion_matrix

import numpy as np

# build pipeline
tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=5,
        max_features=200_000
    )),
    ("clf", LogisticRegression(
        max_iter=300,
        n_jobs=-1,
        class_weight="balanced"   # IMPORTANT for imbalance
    ))
])

# train
tfidf_clf.fit(tr_df["chiefcomplaint"], tr_df["label"])

# validation predictions
val_pred = tfidf_clf.predict(val_df["chiefcomplaint"])


print("Validation Macro-F1:",
      f1_score(val_df["label"], val_pred, average="macro"))

print("Validation Balanced Accuracy:",
      balanced_accuracy_score(val_df["label"], val_pred))

print("\nPer-class performance:")
print(classification_report(val_df["label"], val_pred, digits=3))

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Validation Macro-F1: 0.3773393115248662
Validation Balanced Accuracy: 0.5625430697978299

Per-class performance:
              precision    recall  f1-score   support

           0      0.202     0.673     0.311      2422
           1      0.618     0.464     0.530     14201
           2      0.776     0.468     0.584     24071
           3      0.278     0.612     0.382      3653
           4      0.042     0.596     0.079       235

    accuracy                          0.490     44582
   macro avg      0.383     0.563     0.377     44582
weighted avg      0.650     0.490     0.533     44582



In [8]:
# @title Trying out Different Parameters

for C in [0.1, 1, 5]:
    for ngram in [(1,1),(1,2),(1,3)]:

        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=ngram,
                min_df=5,
                max_features=200_000
            )),
            ("clf", LogisticRegression(
                C=C,
                max_iter=300,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

        pipe.fit(tr_df["chiefcomplaint"], tr_df["label"])
        pred = pipe.predict(val_df["chiefcomplaint"])

        f1 = f1_score(val_df["label"], pred, average="macro")

        print(C, ngram, f1)

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


0.1 (1, 1) 0.36303832492512306


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


0.1 (1, 2) 0.3748916395941275


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


...

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


5 (1, 2) 0.37553795894882996


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


5 (1, 3) 0.3747377985761992


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Testing the Model 

In [21]:
best_tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=5,
        max_features=200_000,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        C=0.1,
        max_iter=1000,
        class_weight="balanced",
        solver="saga"
    ))
])

# ---- train on training split only ----
best_tfidf_clf.fit(tr_df["chiefcomplaint"], tr_df["label"])

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


#### Adult Data Test

In [22]:
# ---- final test evaluation ----
from sklearn.metrics import cohen_kappa_score

test_pred = best_tfidf_clf.predict(test_df["chiefcomplaint"])

print("\nTEST RESULTS (adult)")
print("Test Macro-F1:",
      f1_score(test_df["label"], test_pred, average="macro"))
print("Test Balanced Accuracy:",
      balanced_accuracy_score(test_df["label"], test_pred))
print("Test Quadratic Cohen's Kappa:",
      cohen_kappa_score(test_df["label"], test_pred, weights="quadratic"))

print("\nConfusion Matrix:")
print(confusion_matrix(test_df["label"], test_pred))

print("\nClassification Report:")
print(classification_report(
    test_df["label"],
    test_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[f"Acuity {id2label[i]}" for i in range(5)],
    digits=3
))


TEST RESULTS (adult)
Test Macro-F1: 0.3635129536695416
Test Balanced Accuracy: 0.4913983850788545
Test Quadratic Cohen's Kappa: 0.43423933356129096

Confusion Matrix:
[[ 1502   307   393    91    17]
 [ 3977  5838  3726   697   151]
 [ 2805  3334 13044  3965   802]
 [  206   204  1105  1755   447]
 [   16    16    28    73    83]]

Classification Report:
              precision    recall  f1-score   support

    Acuity 1      0.177     0.650     0.278      2310
    Acuity 2      0.602     0.406     0.485     14389
    Acuity 3      0.713     0.545     0.618     23950
    Acuity 4      0.267     0.472     0.341      3717
    Acuity 5      0.055     0.384     0.097       216

    accuracy                          0.498     44582
   macro avg      0.363     0.491     0.364     44582
weighted avg      0.609     0.498     0.531     44582



#### Pediatric Data Test

In [24]:
test_peds_df = pd.read_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-pedsNHAMCS.csv')
test_peds_df["label"] = test_peds_df["acuity"].map(label2id)

test_peds_df = test_peds_df.dropna(subset=["chiefcomplaint"]).copy()
test_peds_df["chiefcomplaint"] = test_peds_df["chiefcomplaint"].astype(str)
test_peds_df = test_peds_df[test_peds_df["chiefcomplaint"].str.strip() != ""]

test_peds_df = test_peds_df.dropna(subset=["label"])
test_peds_df["label"] = test_peds_df["label"].astype(int)

# ---- final test evaluation ----
test_peds_pred = best_tfidf_clf.predict(test_peds_df["chiefcomplaint"])

print("\nTEST RESULTS (pediatric)")
print("Test Macro-F1:",
      f1_score(test_peds_df["label"], test_peds_pred, average="macro"))
print("Test Balanced Accuracy:",
      balanced_accuracy_score(test_peds_df["label"], test_peds_pred))
print("Test Quadratic Cohen's Kappa:",
      cohen_kappa_score(test_peds_df["label"], test_peds_pred, weights="quadratic"))

print("\nConfusion Matrix:")
print(confusion_matrix(test_peds_df["label"], test_peds_pred))

print("\nClassification Report:")
print(classification_report(
    test_peds_df["label"],
    test_peds_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[f"Acuity {id2label[i]}" for i in range(5)],
    digits=3
))


TEST RESULTS (pediatric)
Test Macro-F1: 0.2257028300206218
Test Balanced Accuracy: 0.271825775696887
Test Quadratic Cohen's Kappa: 0.1342847070481008

Confusion Matrix:
[[   4    1    6   24   14]
 [  38   77  123  178  156]
 [ 146   86  384  757  769]
 [ 100   61  211 1159  901]
 [  11    8   28  151  188]]

Classification Report:
              precision    recall  f1-score   support

    Acuity 1      0.013     0.082     0.023        49
    Acuity 2      0.330     0.135     0.191       572
    Acuity 3      0.511     0.179     0.265      2142
    Acuity 4      0.511     0.477     0.493      2432
    Acuity 5      0.093     0.487     0.156       386

    accuracy                          0.325      5581
   macro avg      0.292     0.272     0.226      5581
weighted avg      0.459     0.325     0.347      5581



## Fine-tuned ClinicalBERT Transformer

### Training the Model

In [28]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import evaluate

model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    # complaints are tiny; 32 is plenty
    return tokenizer(batch["chiefcomplaint"], truncation=True, max_length=32)

# build HF datasets
train_ds = Dataset.from_pandas(tr_df[["chiefcomplaint", "label"]])
val_ds   = Dataset.from_pandas(val_df[["chiefcomplaint", "label"]])
test_ds  = Dataset.from_pandas(test_df[["chiefcomplaint", "label"]])

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])
val_ds   = val_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])
test_ds  = test_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# model: 5 labels (0-4)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

args = TrainingArguments(
    output_dir="triage_text_bioclinicalbert",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    logging_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none",
    fp16=False,
    bf16=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



Map:   0%|          | 0/356653 [00:00<?, ? examples/s]

Map:   0%|          | 0/44582 [00:00<?, ? examples/s]

Map:   0%|          | 0/44582 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [ ]:
trainer.train()

In [30]:
from transformers import AutoModelForSequenceClassification, Trainer
from sklearn.metrics import cohen_kappa_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    "triage_text_bioclinicalbert/checkpoint-22292"
)

trainer = Trainer(model=model,data_collator=data_collator)

pred_output = trainer.predict(test_ds)

logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=1)

kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")

print("Quadratic Cohen's Kappa:", kappa)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Quadratic Cohen's Kappa: 0.5377149789753807


### Testing the Model

#### Adult Data Test

In [ ]:
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score, confusion_matrix, cohen_kappa_score
import numpy as np

# Get predictions
test_output = trainer.predict(test_ds)

test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

print("TEST RESULTS")
print("Macro-F1:", f1_score(test_labels, test_preds, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(test_labels, test_preds))
print("Test Quadratic Cohen's Kappa:",
      cohen_kappa_score(test_labels, test_preds, weights="quadratic"))

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, digits=3))

NameError: name 'trainer' is not defined

#### Pediatric Data Test

In [31]:
# Testing on pediatric data
# Get predictions
label2id = {1:0, 2:1, 3:2, 4:3, 5:4}
id2label = {v:k for k,v in label2id.items()}

test_peds_df = pd.read_csv('/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-pedsNHAMCS.csv')
test_peds_df["label"] = test_peds_df["acuity"].map(label2id)

test_peds_df = test_peds_df.dropna(subset=["chiefcomplaint"])
test_peds_df = test_peds_df[test_peds_df["chiefcomplaint"].str.strip() != ""]

test_peds_ds  = Dataset.from_pandas(test_peds_df[["chiefcomplaint", "label"]])
test_peds_ds  = test_peds_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])

test_peds_output = trainer.predict(test_peds_ds)

test_peds_preds = np.argmax(test_peds_output.predictions, axis=-1)
test_peds_labels = test_peds_output.label_ids

print("TEST RESULTS")
print("Macro-F1:", f1_score(test_peds_labels, test_peds_preds, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(test_peds_labels, test_peds_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(test_peds_labels, test_peds_preds))

print("\nClassification Report:")
print(classification_report(test_peds_labels, test_peds_preds, digits=3))

kappa = cohen_kappa_score(test_peds_labels, test_peds_preds, weights="quadratic")

print("Quadratic Cohen's Kappa:", kappa)

Map:   0%|          | 0/5581 [00:00<?, ? examples/s]

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TEST RESULTS
Macro-F1: 0.29863193728150084
Balanced Accuracy: 0.2961406742319982

Confusion Matrix:
[[   3    2   28   15    1]
 [   2  127  371   72    0]
 [   0   99 1639  402    2]
 [   1   26 1359 1045    1]
 [   1    4  181  199    1]]

Classification Report:
              precision    recall  f1-score   support

           0      0.429     0.061     0.107        49
           1      0.492     0.222     0.306       572
           2      0.458     0.765     0.573      2142
           3      0.603     0.430     0.502      2432
           4      0.200     0.003     0.005       386

    accuracy                          0.504      5581
   macro avg      0.436     0.296     0.299      5581
weighted avg      0.507     0.504     0.471      5581

Quadratic Cohen's Kappa: 0.2871583879356183


In [ ]:
test_peds_ds

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5581
})